In [1]:
queries = [
    # Easy: keyword-based
    {"id": 1, "query": "argan oil hair spray", "type": "keyword"},
    {"id": 2, "query": "volumizing dry shampoo", "type": "keyword"},
    {"id": 3, "query": "sulfate free shampoo", "type": "keyword"},

    # Medium: semantic
    {"id": 4, "query": "something to make frizzy hair smooth", "type": "semantic"},
    {"id": 5, "query": "product that protects hair from heat damage", "type": "semantic"},
    {"id": 6, "query": "gentle cleanser for sensitive scalp", "type": "semantic"},
    {"id": 7, "query": "natural scent that is not overpowering", "type": "semantic"},

    # Complex: require context, filtering, or reasoning
    {"id": 8, "query": "best leave-in conditioner for thick curly hair under $20", "type": "complex"},
    {"id": 9, "query": "what do people say about this product causing hair loss", "type": "complex"},
    {"id": 10, "query": "highly rated hair product that works for both men and women", "type": "complex"},
]

for q in queries:
    print(f"[{q['type'].upper():8}] {q['query']}")

[KEYWORD ] argan oil hair spray
[KEYWORD ] volumizing dry shampoo
[KEYWORD ] sulfate free shampoo
[SEMANTIC] something to make frizzy hair smooth
[SEMANTIC] product that protects hair from heat damage
[SEMANTIC] gentle cleanser for sensitive scalp
[SEMANTIC] natural scent that is not overpowering
[COMPLEX ] best leave-in conditioner for thick curly hair under $20
[COMPLEX ] what do people say about this product causing hair loss
[COMPLEX ] highly rated hair product that works for both men and women


In [11]:
#  BM25 
from rank_bm25 import BM25Okapi
import nltk
import pandas as pd
nltk.download('punkt')
from nltk.tokenize import word_tokenize
result = pd.read_parquet('../data/processed/merged.parquet')

# Build corpus: combine review title + text
corpus = (result['title'].fillna('') + ' ' + result['text'].fillna('')).tolist()
tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=5):
    tokens = word_tokenize(query.lower())
    scores = bm25.get_scores(tokens)
    top_indices = scores.argsort()[::-1][:top_k]
    return result.iloc[top_indices][['product_title', 'text', 'rating']].reset_index(drop=True)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

# Load pre-computed embeddings instead of re-encoding
corpus_embeddings = np.load("../data/processed/embeddings.npy")

print(f"Embeddings shape: {corpus_embeddings.shape}")  # should be (num_docs, 384)

def semantic_search(query, top_k=5):
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, corpus_embeddings)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    return result.iloc[top_indices][['product_title','title', 'text', 'rating']].reset_index(drop=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (701528, 384)


In [ ]:
all_results = {}

for q in queries:
    qid = q['id']
    query_text = q['query']
    all_results[qid] = {
        "query": query_text,
        "type": q['type'],
        "bm25": bm25_search(query_text),
        "semantic": semantic_search(query_text)
    }
    print(f"Done: Q{qid} — {query_text}")

Done: Q1 — argan oil hair spray
Done: Q2 — volumizing dry shampoo
Done: Q3 — sulfate free shampoo
Done: Q4 — something to make frizzy hair smooth
Done: Q5 — product that protects hair from heat damage
Done: Q6 — gentle cleanser for sensitive scalp
Done: Q7 — natural scent that is not overpowering


In [ ]:
chosen = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # adjust to your chosen 5

for qid in chosen:
    q = all_results[qid]
    print(f"\n{'='*60}")
    print(f"Q{qid} [{q['type'].upper()}]: {q['query']}")
    print(f"\n--- BM25 Top 5 ---")
    display(q['bm25'])
    print(f"\n--- Semantic Top 5 ---")
    display(q['semantic'])


Q1 [KEYWORD]: argan oil hair spray

--- BM25 Top 5 ---


,product_title,text,rating
0,"Nadira Organics Virgin Argan Oil for Skin, Fac...","Love this Argan oil, on my second bottle. Use ...",5.0
1,Savanaahs Organic Argan Oil 4 Oz / 120 ML 100%...,I love using Argan oil in my hair it is the pe...,4.0
2,Savanaahs Organic Argan Oil 4 Oz / 120 ML 100%...,I've been fooled by the 'Argan oil' label befo...,5.0
3,Livordo Moroccan Argan Oil Essential Organic C...,My daughter takes every Argan oil I get from m...,5.0
4,Savanaahs Organic Argan Oil 4 Oz / 120 ML 100%...,I love this oil. It is perfect. I used it on...,5.0



--- Semantic Top 5 ---


,product_title,text,rating
0,Cuticle Nippers,The cuticle nipper kept getting stuck. It does...,1.0
1,EcoBrow - All Natural Eyebrow Defining Wax (Pe...,"I'm in love, I received the Liz color as a sam...",5.0
2,New York Biology Vitamin C Serum for Face and ...,"Well, they weren't lying when they said it wor...",5.0
3,Compatible Oral-B Replacement Brush Heads - Va...,Easy Replacement and worked on other device I ...,5.0
4,WAVEBUILDER Seamless Durag- Black,I no longer have to worry about a line going d...,5.0



Q2 [KEYWORD]: volumizing dry shampoo

--- BM25 Top 5 ---


...

,product_title,text,rating
0,"Fleece Ear Warmers Headband for Men & Women, T...",Made my face break out..,2.0
1,Mitchell's Wool Fat Shave Refill Soap,I purchased this shave soap based on the reput...,2.0
2,Salux Nylon Japanese Beauty Skin Bath Wash Clo...,I had a Korean body scrub in Los Angeles and f...,5.0
3,Moresoo 12 Inch Clip in Human Hair Extensions ...,Definitely will keep ordering from this seller...,5.0
4,Bath Scrubber Body Brush Shower Scrubber Back ...,"Loved the brush because of its softness, but t...",4.0



Q10 [COMPLEX]: highly rated hair product that works for both men and women

--- BM25 Top 5 ---


,product_title,text,rating
0,JOOYHOOM Professional Hair Cutting Scissors Se...,Great price for a set of hair cutting kit! Wor...,5.0
1,SilkFeet NightCare Flexible Bladeless Exfoliat...,BEST BEST BEST foot care product,5.0
2,Baby Fresh Powder Perfume Scent - .33oz/10ml R...,The best thing I’ve ever smelled men and women...,5.0
3,Fearless Tape - Sensitive Skin - Women's Doubl...,Good for both men and women.,5.0
4,Cuba Quad 1 Gift Set (4 x 1.17 fl.oz. Eau De T...,Amazing combination of scents and flavors. Uni...,5.0



--- Semantic Top 5 ---


,product_title,text,rating
0,"Gel Toe Separators,XUZOU Toe Straighteners for...",They arrived very quickly and I used them righ...,5.0
1,Organic Activated Black Best Charcoal Natural ...,When you’re looking for a teeth whitener that’...,5.0
2,Healthcom Makeup Bags and Cases,This is a good buy. I'm not sure what to call ...,4.0
3,Francis,"Ok fragrance, but too floral for me. I would ...",3.0
4,Hot Spike Stud Blouse Shirts Collar Neck Tip B...,Cute fell in love with it but just took a long...,5.0
